<a href="https://colab.research.google.com/github/sabithakrishnan/clinical_classifierwith-Biomistral/blob/main/Biomistral.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
!pip install -q -U transformers bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "BioMistral/BioMistral-7B"

# Configure 4-bit quantization to save RAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 46.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.5GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Error during conversion: ReadTimeout('The read operation timed out')


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import torch

# Set the padding token for the tokenizer
tokenizer.pad_token = tokenizer.eos_token

# Function to extract embeddings using BioMistral
def extract_embeddings(texts):
    # Tokenize the texts
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # Get the last hidden states
    # Using the last hidden state and averaging tokens for sentence embedding
    embeddings = outputs.hidden_states[-1].mean(dim=1).to(torch.float32).cpu().numpy()
    return embeddings

# Example dataset: patient symptoms and a binary label (e.g., 1 for "At Risk", 0 for "Not at Risk")
data = {
    "text": [
        "Patient shows high glucose and blurred vision",
        "Patient has normal vitals and no pain",
        "Patient reports severe headache and nausea",
        "Patient has elevated blood pressure and chest pain",
        "Patient is asymptomatic with routine checkup",
        "Patient has mild fever and cough",
        "Patient exhibits confusion and memory loss",
        "Patient shows signs of fatigue and muscle weakness"
    ],
    "label": [1, 0, 1, 1, 0, 0, 1, 1]
}

# 1. Convert text to BioMistral embeddings
X = extract_embeddings(data["text"])
y = np.array(data["label"])

# 2. Split data into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Train Logistic Regression model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 4. Evaluate
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred))

Accuracy: 0.5
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
!pip freeze > requirements.txt